In [ ]:
!pip install -U sentence_transformers chromadb langgraph langchain-openai

# Configurando o RAG

In [40]:
import os
import chromadb

from sentence_transformers import SentenceTransformer
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI

## Adicionando politicas em array

In [ ]:
nome_diretorio = "data/policies/"
politicas = []

for nome_arquivo in os.listdir(nome_diretorio):
    caminho_completo = os.path.join(nome_diretorio, nome_arquivo)
    print(caminho_completo)

    if os.path.isfile(caminho_completo):
        with open(caminho_completo, 'r', encoding="utf-8") as f:
            politicas.append(f.read())

data/policies/politica_reembolso.md
data/policies/acesso_conta.md
data/policies/politica_sla.md


In [23]:
print(politicas)

['# Política de Reembolso e Cancelamento\n\n## 1. Regra Geral de Arrependimento (Prazo de 7 Dias)\nTodos os clientes novos que assinarem qualquer um dos nossos planos (Mensal, Semestral ou Anual) possuem o direito assegurado por lei de solicitar o **reembolso integral** em até 7 (sete) dias corridos após a data da compra. \n- Não é necessário apresentar justificativa.\n- O cancelamento deve ser feito dentro da plataforma ou via ticket formal.\n\n## 2. Cancelamentos de Assinaturas (Pós-prazo de 7 dias)\n- **Plano Mensal:** O cancelamento impede a cobrança do mês seguinte. O cliente continua com acesso até o último dia do ciclo já pago. Não há estorno parcial.\n- **Plano Anual:** O cancelamento após os 7 dias iniciais **não** garante o reembolso do valor integral pago de forma antecipada. O cliente pode cancelar a renovação automática, mas o serviço permanecerá ativo até o fim dos 12 meses contratados.\n  - *Exceção:* Em casos de falha grave sistêmica documentada e provada que impediu o 

## Criando embeddings/vetores

In [ ]:
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

embeddings = model.encode(politicas)

In [22]:
print("Formato da matriz de vetores:", embeddings.shape)
print("Vetor do primeiro texto:", embeddings[0][:5]) 

Formato da matriz de vetores: (3, 384)
Vetor do primeiro texto: [ 0.10730104  0.01454751 -0.13450992 -0.06465343 -0.02914495]


## Criando o banco vetorial

In [ ]:
client = chromadb.PersistentClient(path="meu_banco_vetorial")

collection = client.get_or_create_collection(name="politicas_empresa")
ids = [f"doc_{i}" for i in range(len(politicas))]

metadatas = [{"source": "politica_reembolso_ou_outra", "index": i} for i in range(len(politicas))]
collection.add(
    ids=ids,
    embeddings=embeddings.tolist(),
    documents=politicas,
    metadatas=metadatas
)

## Fazendo consulta de teste

In [24]:
# Fazendo uma query de busca por similaridade
query_texto = "Qual é o prazo de reembolso?"
query_vector = model.encode(query_texto).tolist()

resultados = collection.query(
    query_embeddings=[query_vector],
    n_results=2
)


In [25]:
print("Documentos mais similares encontrados:")
for doc, score in zip(resultados['documents'][0], resultados['distances'][0]):
    print(f"\n- Distância: {score:.4f}\n- Conteúdo: {doc[:150]}...")

Documentos mais similares encontrados:

- Distância: 14.3967
- Conteúdo: # Política de Reembolso e Cancelamento

## 1. Regra Geral de Arrependimento (Prazo de 7 Dias)
Todos os clientes novos que assinarem qualquer um dos no...

- Distância: 20.5338
- Conteúdo: # Política de Acesso e Segurança de Contas

## 1. Redefinição de Senha
- Clientes que esquecerem a senha podem solicitar a recuperação diretamente na ...


In [34]:
def buscar_documentos_para_o_cliente(mensagem_do_ticket):
    vetor_da_mensagem = model.encode(mensagem_do_ticket).tolist()

    resultados = collection.query(
        query_embeddings=[vetor_da_mensagem],
        n_results=1
    )

    documento_encontrado = resultados['documents'][0][0]
    return documento_encontrado

In [35]:
pergunta = "Esqueci minha senha, o que preciso fazer?"
contexto = buscar_documentos_para_o_cliente(pergunta)

In [ ]:
print(prompt)

In [48]:
MINIMAX_API_KEY = "sk-cp-b-OId1P3K06TIkXUpRoRFb27BtALA1XaUDCEPCXUnOn8EIW_raHfyVnoNd19a8l6X5DSZ4vS4ZN-PctuAgc1iNwyBHrndyndIbzaks48UOP7M8ToVJbiq-Q"

llm = ChatOpenAI(
    api_key=MINIMAX_API_KEY,
    base_url="https://api.minimax.io/v1",
    model="Minimax-M3"
)

In [67]:
class AgentState(TypedDict):
    pergunta: str
    contexto_encontrado: str
    resposta_final: str

def gera_prompt(contexto: str, pergunta: str):
    prompt = f""""
    Você é um agente de suporte de nível 1. Sua missão é responder às dúvidas dos clientes de forma educada.

    [REGRA DE OURO]
    Responder a pergunta do usuário baseando-se ÚNICA E EXCLUSIVAMENTE nos documentos do contexto abaixo.
    Se a resposta não estiver lá, diga que não tem a informação e que será necessário transferí-lo para um atendente.
    NÃO INVENTE NADA.

    [CONTEXTO DA EMPRESA]
    {contexto}
    [PERGUNTA DO CLIENTE]
    {pergunta}
    """

    return prompt

def no_buscar_documentos(state: AgentState):
    pergunta = state["pergunta"]
    print("[Nó 1] Buscando no banco vetorial...")

    query_vector = model.encode(pergunta).tolist()
    resultados = collection.query(
        query_embeddings=[query_vector],
        n_results=1
    )

    contexto = resultados['documents'][0][0] if resultados['documents'][0] else "Sem resultados."
    
    # Salva o texto achado no 'estado' para o próximo nó usar
    return {"contexto_encontrado": contexto}

def no_gerar_resposta(state: AgentState):
    print("🤖 [Nó 2] Acordando o Minimax para gerar a resposta...")
    contexto = state["contexto_encontrado"]
    pergunta = state["pergunta"]

    prompt = gera_prompt(contexto, pergunta)

    resposta = llm.invoke(prompt)

    return {"resposta_final": resposta.content}
    

In [68]:
workflow = StateGraph(AgentState)

workflow.add_node("bibliotecario", no_buscar_documentos)
workflow.add_node("atendente", no_gerar_resposta)

workflow.add_edge(START, "bibliotecario")
workflow.add_edge("bibliotecario", "atendente")
workflow.add_edge("atendente", END)

orquestrador = workflow.compile()

In [69]:
ticket_usuario = "Bom dia, eu pedi o cancelamento hoje que é meu 5º dia, eu recebo estorno?"

estado_inicial = {"pergunta": ticket_usuario}

resultado = orquestrador.invoke(estado_inicial)

[Nó 1] Buscando no banco vetorial...
🤖 [Nó 2] Acordando o Minimax para gerar a resposta...


In [70]:
print("\n================== RESULTADO FINAL ==================")
print(resultado["resposta_final"])
print("=====================================================")


================== RESULTADO FINAL ==================
<think>
The customer is asking about a cancellation request on their 5th day. According to the policy in the context document:

Section 1 - "Regra Geral de Arrependimento (Prazo de 7 Dias)":
- All new customers who sign up for any plan (Monthly, Semi-annual or Annual) have the right guaranteed by law to request a full refund within 7 (seven) calendar days after the date of purchase.
- No justification is required.
- The cancellation must be made within the platform or via formal ticket.

So yes, since the customer is on their 5th day, they are within the 7-day window and are entitled to a full refund.

I should respond in Portuguese, politely, and based on the context. I should also mention the steps to request the refund (Section 4 - Como Solicitar).

Let me also think about the timeframe for the refund processing (Section 3), which depends on the payment method. I should ask or mention this.

Let me craft a clear, polite response